# 00 · Building the San Francisco dataset

This notebook walks through `src/vibecheck/filter_sf.py` one step at a time.
`make data` runs the same code from the terminal.

**Pipeline**

1. Download the California metadata (118 MB, 516k businesses).
2. Keep places in San Francisco that are cafes, restaurants or bars → `sf_places.parquet`.
3. Stream the California reviews file (5.35 GB) straight from the URL, keep up to the 50 most
   recent non-empty reviews per SF place → `sf_reviews.parquet`.

Both files are read line by line, so memory stays flat.

In [ ]:
from itertools import islice

import matplotlib.pyplot as plt
import pandas as pd

from vibecheck import filter_sf as f
from vibecheck.plots import GROUP_COLORS, use_style

use_style()
pd.set_option("display.max_colwidth", 80)

## 1. What a raw record looks like

Each line of the metadata file is one business. `MISC` holds Google's attribute tags,
such as `Atmosphere: ["Casual", "Cozy"]`. We'll use those later to check search results.

In [ ]:
f.download(f.META_URL, f.META_PATH)
sample = list(islice(f.iter_json_lines(f.META_PATH), 3))
pd.DataFrame(sample)[["name", "address", "category", "avg_rating", "num_of_reviews", "MISC"]]

## 2. The filter rules

All the knobs live at the top of `filter_sf.py`.

- **In SF**: the address contains "San Francisco, CA", or the coordinates are inside the box and the
  address doesn't name another city. The box catches SF addresses written in other formats;
  the city check stops it from pulling in Daly City.
- **Cafe / restaurant / bar**: a whole-word keyword match on Google's categories, so "bar" doesn't
  match "Barber shop", and "Restaurant supply store" is excluded.

In [ ]:
print("SF box: lat", f.SF_LAT, "lon", f.SF_LON)
for group, words in f.CATEGORY_KEYWORDS.items():
    print(f"{group:>10}: {', '.join(words)}")
print("   exclude:", ", ".join(f.EXCLUDE_KEYWORDS))

In [ ]:
places = f.build_places()
places.head()

## 3. Sanity check: where are the places?

If the filter worked, this should look like San Francisco, with dense clusters downtown,
in the Mission and in the Richmond and Sunset.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for group, color in GROUP_COLORS.items():
    sub = places[places["group"] == group]
    ax.scatter(sub["lon"], sub["lat"], s=8, color=color, alpha=0.7, linewidths=0,
               label=f"{group} ({len(sub):,})")
ax.set(title=f"{len(places):,} SF cafes, restaurants and bars", xlabel="longitude", ylabel="latitude")
ax.set_aspect(1.27)  # ~1/cos(37.8°): makes SF look the right shape
ax.legend(loc="lower left", markerscale=2)
plt.show()

## 4. Reviews

The reviews file is 5.35 GB compressed, too big for most laptops' free disk, so it is streamed
from the URL and only SF rows are kept. This takes 30-45 minutes, so the cell reuses
`sf_reviews.parquet` if it already exists.

Per place we keep the 50 most recent reviews with text. Google-translated reviews keep only
their English part.

In [ ]:
reviews = pd.read_parquet(f.REVIEWS_OUT) if f.REVIEWS_OUT.exists() else f.build_reviews()
print(f"{len(reviews):,} reviews for {reviews['gmap_id'].nunique():,} of {len(places):,} places")
print(f"date range: {reviews['time'].min():%Y-%m} to {reviews['time'].max():%Y-%m}")
reviews.head()